# FZI-AURA - Quickstart

This notebook demonstrates the core SDK workflow:

- loading the dataset and split metadata
- inspecting scenes and samples
- using the flat frame dataset
- checking sparse annotation availability
- loading representative camera, LiDAR, box, and semantic data

Set `FZI_AURA_ROOT` or edit `DATASET_ROOT` below. Metadata and 3D boxes require `base_keyframes`. The optional sensor cells use `camera_keyframes` and `lidar_motion_compensated_keyframes`; they print a skip message when those layers are not installed.

In [1]:
import os
from pathlib import Path

os.environ["FZI_AURA_ROOT"] = "/data/fzi-aura"

DATASET_ROOT = Path(
    os.environ.get(
        "FZI_AURA_ROOT",
        "/path/to/fzi-aura",
    )
)
SPLIT = "train"  # Options: "train", "val", "test", "None" -> all scenes
print(f"Using dataset root: {DATASET_ROOT}")

Using dataset root: /data/fzi-aura


## 1. Load a Scene-Level Dataset

In [2]:
from fzi_aura import FZIAURADataset

scenes = FZIAURADataset(DATASET_ROOT, split=SPLIT)
print(f"Scenes in {SPLIT}: {len(scenes)}")
print(f"First scene ids: {scenes.scene_ids[:5]}")
print(
    f"Available layers: {scenes.available_layers}"
)  # None in our example because we used our dataset export directly without redownloading it from HF.

Scenes in train: 1979
First scene ids: ('2025-05-26-15-56-47|11', '2025-05-26-15-56-47|7', '2025-05-26-15-56-47|8', '2025-06-20-10-08-06|10', '2025-06-20-10-08-06|104')
Available layers: None


## 2. Inspect One Scene

In [3]:
scene = scenes[-1]

print(f"Scene ID:   {scene.scene_id}")
print(f"Name:       {scene.name}")
# print(f"Path:       {scene.path)
print(f"Samples:    {len(scene)}")
print(f"Duration:   {scene.duration_s:.2f} s")
print(f"Cameras:    {scene.available_cameras()}")
print(f"Lidars:     {scene.available_lidars()}")
print(f"Radars:     {scene.available_radars()}")

Scene ID:   2026-06-03-09-57-03|98
Name:       2026-06-03-09-57-03_98
Samples:    200
Duration:   19.95 s
Cameras:    ('front_medium', 'front_tele', 'front_wide', 'left_forward', 'left_rearward', 'rear_wide', 'right_forward', 'right_rearward')
Lidars:     ('aeva_front_center', 'aeva_front_left', 'aeva_front_right', 'aeva_rear_center', 'aeva_side_left', 'aeva_side_right', 'front_left', 'front_right', 'rear_left', 'rear_right', 'top_left', 'top_right')
Radars:     ('front_left', 'front_right', 'rear_center')


## 3. Inspect a Frame

Frames are lazy handles. Paths and metadata are available immediately; images, point clouds, boxes, and labels load only when requested.

In [4]:
frame = scene[0]

print(f"Scene:       {frame.scene_id}")
print(f"Frame index: {frame.frame_index}")
print(f"Timestamp:   {frame.timestamp_ns}")
print(f"Cameras:     {frame.available_cameras()}")
print(f"Lidars MC:   {frame.available_lidars('motion_compensated')}")
print(f"Lidars raw:  {frame.available_lidars('raw')}")
print(f"Radars:      {frame.available_radars()}")
print(f"Annotations frame 0: {frame.annotation_types}")

# Labeling was done in 2hz
frame = scene[1]
print(f"Annotations frame 1: {frame.annotation_types}")

frame = scene[5]
print(f"Annotations frame 5: {frame.annotation_types}")

Scene:       2026-06-03-09-57-03|98
Frame index: 0
Timestamp:   1780482729200000000
Cameras:     ('front_medium', 'front_tele', 'front_wide', 'left_forward', 'left_rearward', 'right_forward', 'right_rearward')
Lidars MC:   ('aeva_front_center', 'aeva_front_left', 'aeva_front_right', 'aeva_rear_center', 'aeva_side_left', 'aeva_side_right', 'front_left', 'front_right', 'rear_left', 'rear_right', 'top_left', 'top_right')
Lidars raw:  ('aeva_front_center', 'aeva_front_left', 'aeva_front_right', 'aeva_rear_center', 'aeva_side_left', 'aeva_side_right', 'front_left', 'front_right', 'rear_left', 'rear_right', 'top_left', 'top_right')
Radars:      ('front_left', 'front_right', 'rear_center')
Annotations frame 0: ('boxes_3d', 'semantic_lidar')
Annotations frame 1: ()
Annotations frame 5: ('boxes_3d', 'semantic_lidar')


## 4. Flat Frame Dataset

`scene.frames(...)` creates a scene-local `FrameDataset`. Use `scenes.as_frames(...)` for the same filters across the full split; full-split indexing scans every selected scene.

In [5]:
all_frames = scene.frames(sample_filter="all")
box_frames = scene.frames(sample_filter="boxes_3d")
sensor_frames = scene.frames(
    sample_filter="sensor_available",
    require_cameras=["front_medium"],
    require_lidars=["top_left"],
)
semantic_frames = scene.frames(
    sample_filter="semantic_lidar",
    require_lidars=["top_left"],
    require_semantics_for=["top_left"],
)

print(f"All frames:       {len(all_frames)}")
print(f"3D box frames:    {len(box_frames)}")
print(f"Camera + LiDAR:   {len(sensor_frames)}")
print(f"Semantic frames:  {len(semantic_frames)}")

# row = all_frames.to_dict(0)
# row

All frames:       200
3D box frames:    40
Camera + LiDAR:   200
Semantic frames:  40


## 5. Sparse Annotation Keyframes

The public export contains 10 Hz samples, but annotations are sparse (2 Hz). Use filters when you need labeled samples for training.

In [6]:
print(f"Scene-local 3D box keyframes: {scene.detection_keyframe_indices()}")
print(f"Scene-local semantic keyframes: {scene.semantic_keyframe_indices()}")

if len(box_frames):
    labeled = box_frames[0]
    boxes = labeled.load_boxes(frame="base_link")
    print(
        f"First box frame: scene={labeled.scene_id}, frame={labeled.frame_index}, boxes={len(boxes)}"
    )
else:
    print("No 3D box keyframes are available in this selection.")

Scene-local 3D box keyframes: [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100, 105, 110, 115, 120, 125, 130, 135, 140, 145, 150, 155, 160, 165, 170, 175, 180, 185, 190, 195]
Scene-local semantic keyframes: [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100, 105, 110, 115, 120, 125, 130, 135, 140, 145, 150, 155, 160, 165, 170, 175, 180, 185, 190, 195]
First box frame: scene=2026-06-03-09-57-03|98, frame=0, boxes=1


## 6. Load Representative Data

The frame objects above are lazy. Data is read only by the corresponding `load_*` call. Sensor-dependent examples remain optional so the same notebook works with selective downloads.

In [7]:
if len(sensor_frames):
    sensor_frame = sensor_frames[0]
    image = sensor_frame.load_camera("front_medium")
    cloud = sensor_frame.load_lidar("top_left", stage="motion_compensated")
    print(f"Camera RGB: {image.shape} {image.dtype}")
    print(f"LiDAR XYZ: {cloud.xyz.shape} {cloud.xyz.dtype}")
    print(f"LiDAR fields: {cloud.field_names}")
else:
    print(
        "Camera/LiDAR example skipped: install camera_keyframes and lidar_motion_compensated_keyframes."
    )

if len(semantic_frames):
    semantic_frame = semantic_frames[0]
    semantic_cloud, semantic_labels = semantic_frame.load_lidar_semantic_pair(
        "top_left", stage="motion_compensated"
    )
    print(
        f"Semantic pair: points={semantic_cloud.xyz.shape}, "
        f"semantic={semantic_labels.semantic_id.shape}, "
        f"instance={semantic_labels.instance_id.shape}"
    )
else:
    print(
        "Semantic example skipped: compatible top_left labels and LiDAR are not installed."
    )

Camera RGB: (1200, 1920, 3) uint8
LiDAR XYZ: (76587, 3) float64
LiDAR fields: ('x', 'y', 'z', 't', 'reflectivity', 'ring', 'range')
Semantic pair: points=(76587, 3), semantic=(76587,), instance=(76587,)
